In [ ]:
import argparse
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch import datasets, models, transforms

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: torch.Tensor = None, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        if self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


def compute_class_weights(targets, num_classes: int) -> torch.Tensor:
    counts = np.bincount(targets, minlength=num_classes)
    counts = np.clip(counts, 1, None)  # avoid divide-by-zero for empty classes
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_transforms(img_size: int = 224, train: bool = True) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.Resize((img_size, img_size)),
                transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.3),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
                transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
                # RandomErasing after ToTensor emulates partial occlusion (branches, other vehicles)
                transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


def get_dataloaders(data_dir: str, img_size: int, batch_size: int, num_workers: int, use_weighted_sampler: bool):
    train_dir = Path(data_dir) / "train"
    val_dir = Path(data_dir) / "val"

    train_ds = datasets.ImageFolder(train_dir, transform=build_transforms(img_size, train=True))
    val_ds = datasets.ImageFolder(val_dir, transform=build_transforms(img_size, train=False))

    num_classes = len(train_ds.classes)
    train_targets = [label for _, label in train_ds.samples]
    class_weights = compute_class_weights(train_targets, num_classes)

    if use_weighted_sampler:
        sample_weights = [class_weights[label].item() for label in train_targets]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, train_ds.classes, class_weights


# ---------------------------------------------------------------------------
# 3. Model -- MobileNetV3 backbone with a custom classification head
# ---------------------------------------------------------------------------


def build_mobilenetv3(num_classes: int, variant: str = "small", pretrained: bool = True, freeze_backbone: bool = False) -> nn.Module:
    if variant == "small":
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)
    elif variant == "large":
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.mobilenet_v3_large(weights=weights)
    else:
        raise ValueError("variant must be 'small' or 'large'")

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

def train_one_epoch(model, loader, optimizer, criterion, device) -> tuple:
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, class_names=None) -> tuple:
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    report = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    return running_loss / total, correct / total, report, cm


def fit(args: argparse.Namespace):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    )
    print(f"Using device: {device}")

    train_loader, val_loader, class_names, class_weights = get_dataloaders(
        args.data_dir, args.img_size, args.batch_size, args.num_workers,
        use_weighted_sampler=(args.sampler == "weighted"),
    )
    num_classes = len(class_names)
    class_weights = class_weights.to(device)

    model = build_mobilenetv3(num_classes, variant=args.variant, pretrained=True).to(device)

    if args.loss == "focal":
        criterion = FocalLoss(gamma=args.focal_gamma, weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    Path(args.output_dir).mkdir(parents=True, exist_ok=True)
    best_acc = 0.0
    final_report = ""

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, report, cm = evaluate(model, val_loader, criterion, device, class_names)
        scheduler.step()
        elapsed = time.time() - t0
        final_report = report

        print(
            f"[Epoch {epoch:02d}/{args.epochs}] "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({elapsed:.1f}s)"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            ckpt_path = Path(args.output_dir) / f"mobilenetv3_{args.variant}_best.pt"
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_names": class_names,
                    "img_size": args.img_size,
                    "variant": args.variant,
                    "val_acc": val_acc,
                },
                ckpt_path,
            )
            print(f"  -> new best checkpoint saved to {ckpt_path} (val_acc={val_acc:.4f})")

    print("\nFinal validation classification report:\n", final_report)
    print(f"Best val accuracy: {best_acc:.4f}")
    return model, class_names

def load_model_for_inference(checkpoint_path: str, device: torch.device = None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device)
    model = build_mobilenetv3(len(ckpt["class_names"]), variant=ckpt["variant"], pretrained=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()
    return model, ckpt["class_names"], ckpt["img_size"], device


@torch.no_grad()
def predict(model, class_names, img_size, device, pil_image, min_confidence: float = 0.5):
    transform = build_transforms(img_size, train=False)
    tensor = transform(pil_image).unsqueeze(0).to(device)
    logits = model(tensor)
    probs = F.softmax(logits, dim=1)
    conf, pred_idx = probs.max(dim=1)
    conf = conf.item()
    label = class_names[pred_idx.item()]
    if conf < min_confidence:
        return None, conf
    return label, conf


def benchmark_fps(model, device, img_size: int = 224, num_iters: int = 100) -> tuple:
    model.eval()
    dummy = torch.randn(1, 3, img_size, img_size).to(device)
    with torch.no_grad():
        for _ in range(10):
            model(dummy)  # warm-up
        start = time.time()
        for _ in range(num_iters):
            model(dummy)
        elapsed = time.time() - start
    fps = num_iters / elapsed
    latency_ms = (elapsed / num_iters) * 1000
    print(f"Inference latency: {latency_ms:.2f} ms/frame | Throughput: {fps:.1f} FPS")
    return fps, latency_ms

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module = None):
        self.model = model
        self.target_layer = target_layer or model.features[-1]
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, input_tensor: torch.Tensor, class_idx: int = None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        score = output[:, class_idx]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


# ---------------------------------------------------------------------------
# 7. Export -- TorchScript + CoreML, mirroring the report's own deployment
#    snippet (coremltools.convert on a traced model, integrated via
#    CoreML into the iOS app). For Android/TFLite, export the TorchScript
#    model to ONNX first, then convert ONNX -> TensorFlow -> TFLite.
# ---------------------------------------------------------------------------


def export_torchscript(model: nn.Module, img_size: int, output_path: str):
    model.eval()
    example = torch.randn(1, 3, img_size, img_size)
    traced = torch.jit.trace(model, example)
    traced.save(output_path)
    print(f"Saved TorchScript model to {output_path}")
    return traced


def export_coreml(traced_model, img_size: int, output_path: str):
    """Requires: pip install coremltools"""
    import coremltools as ct

    mlmodel = ct.convert(
        traced_model,
        inputs=[ct.ImageType(name="input", shape=(1, 3, img_size, img_size))],
    )
    mlmodel.save(output_path)
    print(f"Saved CoreML model to {output_path}")


# ---------------------------------------------------------------------------
# 8. CLI entry point
# ---------------------------------------------------------------------------


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train MobileNetV3 for VISTA traffic sign/signal classification")
    parser.add_argument("--data-dir", type=str, required=True,
                         help="Root dir with train/ and val/ subfolders, each in ImageFolder format "
                              "(one subfolder per class).")
    parser.add_argument("--output-dir", type=str, default="./outputs/checkpoints")
    parser.add_argument("--variant", type=str, default="small", choices=["small", "large"])
    parser.add_argument("--img-size", type=int, default=224)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--epochs", type=int, default=25)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--loss", type=str, default="focal", choices=["focal", "weighted_ce"])
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--sampler", type=str, default="weighted", choices=["weighted", "random"])
    parser.add_argument("--num-workers", type=int, default=4)
    return parser


def main():
    args = build_arg_parser().parse_args()
    fit(args)


if __name__ == "__main__":
    main()

    # --- Example post-training usage (uncomment / adapt as needed) ---
    #
    # model, class_names, img_size, device = load_model_for_inference(
    #     "./outputs/checkpoints/mobilenetv3_small_best.pt"
    # )
    # benchmark_fps(model, device, img_size)
    #
    # from PIL import Image
    # crop = Image.open("./sample_signal_crop.jpg").convert("RGB")
    # label, confidence = predict(model, class_names, img_size, device, crop, min_confidence=0.6)
    # print(label, confidence)
    #
    # traced = export_torchscript(model, img_size, "./outputs/mobilenetv3.pt")
    # export_coreml(traced, img_size, "./outputs/MobileNetV3Traffic.mlpackage")